[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gauravs19/iiot-predictive-maintenance/blob/main/notebooks/anomaly_detection.ipynb)

# 🚨 Anomaly Detection — a beginner's guide

This notebook is **independent** from the predictive-maintenance one — you can read it
on its own. It assumes minimal Machine-Learning (ML) knowledge and explains every idea
and abbreviation as it appears.

## 1 · The big picture — what's an "anomaly"?

An **anomaly** is something that doesn't fit the usual pattern — the odd one out. Like
spotting the one kid wearing a heavy winter coat on a hot summer day: you don't need a
rulebook, it just looks *wrong* compared to everyone else.

In the predictive-maintenance notebook we had **labels** — we were *told* which
machines failed. But in real factories, **failures are rare and labelling every sensor
reading is impractical.** So we often have lots of "normal" data and no answer key.

That's where **anomaly detection** shines. It's a form of **unsupervised learning**:

| | Supervised learning | Unsupervised learning |
|---|---|---|
| Answer key (labels)? | Yes | **No** |
| Goal | learn input → known answer | learn what "normal" looks like, then flag the odd ones |
| Everyday analogy | studying with an answer sheet | noticing your car sounds "off" without being told what's wrong |

**The plan:** learn the shape of *healthy* machine data, then flag any reading that
looks too different. We'll use the NASA engine data again, and try **two** methods so
you can compare them.

## The tools (libraries) we'll use — in plain English

A **library** is a bundle of ready-made code someone else wrote so we don't have to.
Here are the ones this notebook uses:

| Library (short name) | What it does | Everyday analogy |
|---|---|---|
| **pandas** (`pd`) | Works with data in **tables** (rows & columns). A table is called a **DataFrame** (`df`). | A smart spreadsheet |
| **NumPy** (`np`) — *Numerical Python* | Fast maths on big lists of numbers (called **arrays**). | A super-fast calculator |
| **Matplotlib** (`plt`) & **Seaborn** (`sns`) | Draw charts and graphs. | Crayons for data |
| **scikit-learn** (`sklearn`) | A toolbox of ready-made ML models and helpers. | A box of LEGO machines |
| **PyTorch** (`torch`) | Builds **neural networks** (brain-inspired models). | LEGO for building a tiny brain |

> Whenever you see `pd.something` it means "use the pandas library to do something".

## 2 · Set up + import tools

In [ ]:
# === Environment bootstrap — works on your laptop AND on Google Colab ========
# (Colab is a free website that runs Python notebooks in your browser, with a
#  free GPU. "GPU" = Graphics Processing Unit, a chip that makes ML training fast.)
import sys, os
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    # On Colab the machine starts empty, so we download ("clone") the project
    # and install the libraries it needs.
    !git clone -q https://github.com/gauravs19/iiot-predictive-maintenance.git
    %cd iiot-predictive-maintenance
    !pip install -q -r requirements.txt

# Add the project folder to Python's search path so `from src import ...` works.
def _find_repo_root(start="."):
    p = os.path.abspath(start)
    while p != os.path.dirname(p):
        if os.path.isdir(os.path.join(p, "src")):
            return p
        p = os.path.dirname(p)
    raise RuntimeError("repo root (folder containing src/) not found")
ROOT = _find_repo_root()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
print("Setup done. Running on", "Colab" if IN_COLAB else "your local machine.")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

from src import data, features, models, utils
print("All tools imported successfully ✅")

## 3 · The data, and what "normal" means here

We use NASA's **C-MAPSS** engine data (C-MAPSS = *Commercial Modular Aero-Propulsion
System Simulation*, the software that generated it). Each row is one **cycle** (time
step) of one engine, with 21 sensor readings.

We don't *train* on failure labels. But to **test** whether our detector works, we
quietly use the engine ages to define:
- **Healthy** = readings from early in an engine's life (lots of life left).
- **Degraded** = readings from just before failure (these *should* look anomalous).

The detectors only ever *learn* from healthy data — exactly like a real deployment
where you fit on normal operation and watch for drift.

In [ ]:
cm = data.load_cmapss("FD001")
df = features.add_rolling_features(
    features.add_rul(cm["train"], clip=125), features.feature_columns(cm["train"]))
cols = features.feature_columns(cm["train"])

healthy = df[df["rul"] >= 100]   # plenty of life left
degraded = df[df["rul"] <= 20]   # near failure
print(f"healthy rows: {len(healthy):,}   degraded rows: {len(degraded):,}")
print("One healthy engine-cycle record:")
display(healthy[cols].iloc[[0]].T)

## 4 · See the difference first

Before any model, let's *look*. We overlay the distribution of a few sensors for
healthy vs degraded readings. A **distribution** shows which values are common (tall)
vs rare (short). Where the red (degraded) and green (healthy) humps **separate**, an
anomaly detector will have an easy time.

In [ ]:
show = ["sensor_2", "sensor_4", "sensor_11", "sensor_15"]
fig, axes = plt.subplots(2, 2, figsize=(13, 7))
for ax, s in zip(axes.ravel(), show):
    ax.hist(healthy[s], bins=40, alpha=0.6, density=True, color="#4c9f70", label="healthy")
    ax.hist(degraded[s], bins=40, alpha=0.6, density=True, color="#d1495b", label="degraded")
    ax.set_title(s); ax.legend()
fig.suptitle("Healthy (green) vs degraded (red) sensor distributions", y=1.02)
fig.tight_layout(); plt.show()

### Two sensors at once

Here we plot two sensors against each other, colouring each point by RUL (Remaining
Useful Life — cycles left). Healthy points (bright) cluster in one region; as engines
degrade (dark), they drift away. Anomaly detection is, visually, "find the points far
from the bright cluster."

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
sample = df.sample(4000, random_state=0)
sc = ax.scatter(sample["sensor_4"], sample["sensor_11"],
                c=sample["rul"], cmap="viridis", s=10, alpha=0.6)
ax.set_xlabel("sensor_4"); ax.set_ylabel("sensor_11")
ax.set_title("Engine states coloured by RUL (dark = near failure)")
plt.colorbar(sc, label="RUL (cycles left)"); plt.show()

### Standardise before modelling

As in any ML with mixed scales, we **standardise** the sensors (rescale each to mean 0,
spread 1) so no single large-numbered sensor dominates. We learn the scale from
**healthy** data, then apply it to both groups.

In [ ]:
scaler = utils.Standardizer().fit(healthy[cols].to_numpy("float32"))
Xh = scaler.transform(healthy[cols].to_numpy("float32"))   # healthy
Xd = scaler.transform(degraded[cols].to_numpy("float32"))  # degraded
print("Standardised:", Xh.shape[0], "healthy and", Xd.shape[0], "degraded rows.")

## 5 · Method 1 — Isolation Forest

**Intuition:** imagine separating one specific person from a crowd by asking random
yes/no questions ("taller than 1.7 m? wearing red?"). An *unusual* person gets singled
out in just a few questions; an average person takes many. An **Isolation Forest**
builds lots of random question-trees and measures how *quickly* each point gets
isolated. Quick to isolate = **anomaly**.

(It's called a "forest" because, like the Random Forest earlier, it uses many random
trees. "Isolation" because it scores how easily a point is *isolated* from the rest.)

We fit it on healthy data, then score everything. We flip the sign so that
**higher score = more anomalous** (easier to read).

In [ ]:
from sklearn.ensemble import IsolationForest
iso = IsolationForest(n_estimators=200, contamination=0.05, random_state=42)
iso.fit(Xh)   # learn what "healthy" looks like

score_h = -iso.decision_function(Xh)   # anomaly score for healthy rows
score_d = -iso.decision_function(Xd)   # ... for degraded rows
print(f"average score  healthy: {score_h.mean():.3f}   degraded: {score_d.mean():.3f}")

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(score_h, bins=40, alpha=0.6, density=True, color="#4c9f70", label="healthy")
ax.hist(score_d, bins=40, alpha=0.6, density=True, color="#d1495b", label="degraded")
ax.set_title("Isolation Forest anomaly scores (higher = more abnormal)")
ax.set_xlabel("anomaly score"); ax.legend(); plt.show()

## 6 · Method 2 — Autoencoder (a neural network)

An **Autoencoder (AE)** is a neural network with an hourglass shape. It does two jobs:
- **Encode**: squeeze the input down into a tiny summary (the "bottleneck").
- **Decode**: rebuild the original from that tiny summary.

We train it **only on healthy data**, so it becomes an expert at rebuilding *normal*
readings. The trick: when we later show it a *degraded* reading it has never learned,
it rebuilds it **badly**. We measure how bad with the **reconstruction error** (how
different the rebuilt version is from the original) — and a big error = anomaly.

Analogy: someone who has only ever drawn cats can redraw any cat from a glance, but
asked to redraw a *dragon* they've never seen, they'll get it very wrong — and that
"wrongness" is the alarm.

We train it to minimise **MSE** = *Mean Squared Error* (the average squared difference
between input and rebuild). The loss curve below should fall as it learns.

In [ ]:
ae = models.AutoEncoder(n_features=Xh.shape[1], latent=8)  # latent=8 = bottleneck size
history = models.train_autoencoder(ae, Xh, epochs=30, batch_size=256)
utils.plot_loss(history, "Autoencoder learning to rebuild healthy data"); plt.show()

In [ ]:
err_h = models.reconstruction_error(ae, Xh)   # rebuild error on healthy
err_d = models.reconstruction_error(ae, Xd)   # rebuild error on degraded
print(f"average rebuild error  healthy: {err_h.mean():.4f}   degraded: {err_d.mean():.4f}")

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(err_h, bins=40, alpha=0.6, density=True, color="#4c9f70", label="healthy")
ax.hist(err_d, bins=40, alpha=0.6, density=True, color="#d1495b", label="degraded")
ax.set_title("Autoencoder reconstruction error (higher = more abnormal)")
ax.set_xlabel("reconstruction error"); ax.legend(); plt.show()

## 7 · Score the detectors fairly

The histograms look promising, but let's measure it properly. We treat "degraded" as
the thing we want to catch and compute **ROC-AUC** for each detector's scores.

- **ROC** = Receiver Operating Characteristic (an old radar term — the name isn't
  important).
- **AUC** = Area Under the Curve: a single number from **0.5** (useless, coin-flip) to
  **1.0** (perfect). It answers: "pick one degraded and one healthy row at random — how
  often does the detector score the degraded one as more abnormal?"

This is a *fair* test because the detectors never saw the health labels — we only use
them now, to grade.

In [ ]:
from sklearn.metrics import roc_auc_score
labels = np.r_[np.zeros(len(Xh)), np.ones(len(Xd))]   # 0 = healthy, 1 = degraded
auc_iso = roc_auc_score(labels, np.r_[score_h, score_d])
auc_ae  = roc_auc_score(labels, np.r_[err_h, err_d])
print(f"Isolation Forest  AUC = {auc_iso:.3f}")
print(f"Autoencoder       AUC = {auc_ae:.3f}")
print("\n(1.0 = perfect separation, 0.5 = no better than guessing)")

## 8 · The real test: does the alarm rise as an engine dies?

The most convincing demo: follow **one engine** across its whole life and plot its
anomaly score over time. A good detector stays quiet while the engine is healthy and
**climbs** as failure approaches — an early-warning siren.

In [ ]:
e = df[df["unit"] == 1].sort_values("cycle")
Xe = scaler.transform(e[cols].to_numpy("float32"))
err_e = models.reconstruction_error(ae, Xe)

fig, ax1 = plt.subplots(figsize=(10, 4))
ax1.plot(e["cycle"], err_e, color="#d1495b", label="anomaly score")
ax1.set_xlabel("cycle (age)"); ax1.set_ylabel("anomaly score", color="#d1495b")
ax2 = ax1.twinx()
ax2.plot(e["cycle"], e["rul"], color="#3b7dd8", alpha=0.6, label="RUL")
ax2.set_ylabel("RUL — cycles left", color="#3b7dd8")
ax1.set_title("Engine #1: the anomaly alarm rises as life (RUL) runs out")
plt.show()

## 9 · Turning a score into an alarm — the threshold

A continuous score is nice, but operators need a yes/no **alarm**. We pick a
**threshold**: score above it → raise the alarm. A common, label-free choice is a high
**percentile** of the healthy scores — e.g. the 99th percentile means "only the most
abnormal 1% of *normal* operation would trip it."

This is a balancing act:
- Threshold **too low** → catches everything, but lots of **false alarms** (crying wolf).
- Threshold **too high** → few false alarms, but you **miss** real problems.

This is the same precision/recall trade-off from the other notebook, in disguise.

In [ ]:
threshold = np.percentile(err_h, 99)
caught = (err_d > threshold).mean()
false_alarms = (err_h > threshold).mean()
print(f"threshold (99th percentile of healthy) = {threshold:.4f}")
print(f"degraded readings correctly flagged    = {100*caught:.1f}%")
print(f"false alarms on healthy readings        = {100*false_alarms:.1f}%")

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(err_h, bins=40, alpha=0.6, density=True, color="#4c9f70", label="healthy")
ax.hist(err_d, bins=40, alpha=0.6, density=True, color="#d1495b", label="degraded")
ax.axvline(threshold, color="black", ls="--", label="alarm threshold")
ax.set_title("Where we draw the alarm line"); ax.set_xlabel("reconstruction error")
ax.legend(); plt.show()

## 10 · Recap

You built **two** anomaly detectors **without any failure labels**:

1. **Isolation Forest** — flags points that are quick to isolate with random questions.
2. **Autoencoder** — learns to rebuild *normal* data; big rebuild error = anomaly.

You also learned to grade them fairly (ROC-AUC), watch the alarm rise over an engine's
life, and set a sensible alarm threshold (trading false alarms against missed
detections).

**Why this matters:** unsupervised anomaly detection is often the *only* practical
option on real equipment, where labelled failures are scarce. It catches problems you
were never explicitly taught to look for.

## 📖 Glossary — every abbreviation in one place

| Short | Full term | Meaning in one line |
|---|---|---|
| ML | Machine Learning | Teaching a computer to find patterns from examples instead of fixed rules. |
| IIoT | Industrial Internet of Things | Factory machines fitted with sensors that send data. |
| IoT | Internet of Things | Everyday devices connected to the internet. |
| PdM | Predictive Maintenance | Fixing a machine *just before* it breaks, using data. |
| RUL | Remaining Useful Life | How many cycles/hours a machine has left before failure. |
| LSTM | Long Short-Term Memory | A neural network that remembers earlier steps in a sequence. |
| RNN | Recurrent Neural Network | A network family that reads data step by step (LSTM is one). |
| MSE | Mean Squared Error | Average of the squared mistakes (how wrong, on average). |
| RMSE | Root MSE | Square root of MSE — error in the original units (e.g. cycles). |
| ROC | Receiver Operating Characteristic | A curve showing a classifier's trade-offs. |
| AUC | Area Under the Curve | One number (0–1) summarising the ROC curve; higher = better. |
| AE | Autoencoder | A network that learns to compress then rebuild data. |
| df | DataFrame | A table of data in pandas. |
| rpm | revolutions per minute | How fast something spins. |
| K | Kelvin | A temperature unit (0 K = absolute zero; room ≈ 300 K). |
| Nm | Newton-metre | A unit of torque (twisting force). |
| UCI | Univ. of California, Irvine | A famous free dataset repository. |
| NASA | (US space agency) | Source of the engine dataset. |
| C-MAPSS | Commercial Modular Aero-Propulsion System Simulation | NASA software that simulated the engine data. |